In [42]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()

True

## PART 1 — Getting Started with OpenAI

**Task 1: OpenAI Setup & Basic Prompt**
1. Set up OpenAI API key.
2. Call an OpenAI chat model.
3. Send a simple prompt and print the response.

In [43]:
if os.getenv("OPENAI_API_KEY") is None:
    raise ValueError("OPENAI_API_KEY is not set")
else:
    print("OPENAI_API_KEY is set")
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


OPENAI_API_KEY is set


In [44]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

response = llm.invoke("What is the capital of France?")
print(response)

content='The capital of France is Paris.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2d78c43f25', 'id': 'chatcmpl-EKeXQEs5NxPwKLbpVDhbpyFhjcMSt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a07056-3b51-7703-95f2-20eca30eacff-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## PART 2 — Retriever-Based RAG (Wikipedia + Vector Store)

**Task 2: Wikipedia Retriever**
1. Use LangChain's WikipediaRetriever.
2. Retrieve documents for a user query.
3. Print retrieved content.

In [45]:
from langchain_community.retrievers import WikipediaRetriever

In [46]:
# Initialize the retriever with custom parameters
retriever = WikipediaRetriever(
    lang="en",                 
    top_k_results=3,
)

In [47]:
query = "How LLM works?"
results = retriever.invoke(query)
print(results)


[Document(metadata={'title': 'Large language model', 'summary': "A large language model (LLM) is an AI model (typically a neural network) trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts. They are the basis for many modern chatbots, such as ChatGPT, Claude, Gemini, Grok, and DeepSeek.\nLLMs are typically based on transformer architecture. Generative pre-trained transformers (GPTs) are a type of LLM that is pre-trained to predict the next word. GPTs are then often fine-tuned to follow instructions and to behave as assistants.\nBiased or inaccurate training data can make an LLM's output less reliable. Benchmark evaluations for LLMs attempt to measure model reasoning, factual accuracy, alignment, and safety.", 'source': 'https://en.wikipedia.org/wiki/Large_language_model'}, page_content='A large language model (LLM) is an AI model (typically a neural 


**Task 3: Vector Store Retriever**
1. Load documents (Wikipedia or custom text).
2. Create embeddings using OpenAI.
3. Store embeddings in a vector store (FAISS or Chroma).
4. Perform similarity search using a retriever.

In [48]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [49]:
# sample Documents
docs = [
    Document(
        page_content="""
        AcmeMart is an online retail company offering electronics, home appliances,
        fashion, and grocery products. Customers can place orders through the website
        or mobile application. Orders are processed through regional fulfillment centers.
        The company currently operates in India, Singapore, and the UAE.
        """,
        metadata={
            "source": "company_overview.txt",
            "category": "company",
            "author": "AcmeMart",
            "date": "2026-01-10"
        }
    ),

    Document(
        page_content="""
        Customers can cancel an order before it enters the shipped status.
        To cancel an order, customers should open My Orders, select the relevant order,
        and click Cancel Order. If the order has already been shipped, cancellation is
        not available and the customer must initiate a return after delivery.
        """,
        metadata={
            "source": "order_cancellation.txt",
            "category": "orders",
            "author": "Customer Support",
            "date": "2026-02-01"
        }
    ),

    Document(
        page_content="""
        AcmeMart offers a 30-day return policy for most products. Products must be
        unused and returned with the original packaging, accessories, and invoice.
        Certain categories such as groceries, personal-care products, and customized
        products may not be eligible for return.
        """,
        metadata={
            "source": "return_policy.txt",
            "category": "returns",
            "author": "Customer Support",
            "date": "2026-02-05"
        }
    ),

    Document(
        page_content="""
        Refunds are initiated after the returned product passes quality inspection.
        Credit-card refunds normally take 5-7 business days after approval.
        UPI refunds generally take 2-3 business days. If a refund has not been received
        after the expected period, customers should contact customer support with
        their order ID.
        """,
        metadata={
            "source": "refund_policy.txt",
            "category": "payments",
            "author": "Finance Team",
            "date": "2026-02-10"
        }
    ),

    Document(
        page_content="""
        Standard delivery usually takes 3-5 business days for major cities and
        5-8 business days for other serviceable locations. Express delivery is
        available for selected products and locations and typically takes 1-2
        business days. Delivery estimates are shown during checkout.
        """,
        metadata={
            "source": "shipping_policy.txt",
            "category": "shipping",
            "author": "Logistics Team",
            "date": "2026-02-15"
        }
    ),

    Document(
        page_content="""
        AcmeMart provides customer support through live chat, email, and phone.
        Live chat is available from 8 AM to 10 PM IST every day. Phone support is
        available from 9 AM to 8 PM IST Monday through Saturday. Customers should
        provide their order ID when contacting support about an existing order.
        """,
        metadata={
            "source": "customer_support.txt",
            "category": "support",
            "author": "Customer Support",
            "date": "2026-02-20"
        }
    ),

    Document(
        page_content="""
        AcmeMart uses three main payment methods: credit/debit cards, UPI, and
        net banking. Cash on Delivery is available for eligible products and
        locations. Some high-value products may require online payment and may
        not support Cash on Delivery.
        """,
        metadata={
            "source": "payment_methods.txt",
            "category": "payments",
            "author": "Finance Team",
            "date": "2026-02-25"
        }
    ),

    Document(
        page_content="""
        AcmeMart rewards customers through the Acme Rewards program. Customers
        earn 1 reward point for every ₹100 spent on eligible purchases. Reward
        points can be redeemed during checkout. Points expire 12 months after
        they are earned.
        """,
        metadata={
            "source": "rewards_program.txt",
            "category": "loyalty",
            "author": "Marketing Team",
            "date": "2026-03-01"
        }
    ),

    Document(
        page_content="""
        Premium customers receive free standard shipping, early access to selected
        sales, and priority customer support. Premium membership costs ₹999 per year.
        The membership fee is non-refundable after the 7-day cancellation period.
        Premium benefits apply only to eligible products and locations.
        """,
        metadata={
            "source": "premium_membership.txt",
            "category": "membership",
            "author": "Marketing Team",
            "date": "2026-03-05"
        }
    ),

    Document(
        page_content="""
        AcmeMart's privacy policy states that customer information is used to process
        orders, provide customer support, prevent fraud, and improve services.
        Customers can request access to or deletion of their personal information
        by contacting the privacy team. Payment card details are processed through
        authorized payment providers and are not stored directly by AcmeMart.
        """,
        metadata={
            "source": "privacy_policy.txt",
            "category": "privacy",
            "author": "Legal Team",
            "date": "2026-03-10"
        }
    )
]

docs_text = [doc.page_content for doc in docs]
print(docs_text)

['\n        AcmeMart is an online retail company offering electronics, home appliances,\n        fashion, and grocery products. Customers can place orders through the website\n        or mobile application. Orders are processed through regional fulfillment centers.\n        The company currently operates in India, Singapore, and the UAE.\n        ', '\n        Customers can cancel an order before it enters the shipped status.\n        To cancel an order, customers should open My Orders, select the relevant order,\n        and click Cancel Order. If the order has already been shipped, cancellation is\n        not available and the customer must initiate a return after delivery.\n        ', '\n        AcmeMart offers a 30-day return policy for most products. Products must be\n        unused and returned with the original packaging, accessories, and invoice.\n        Certain categories such as groceries, personal-care products, and customized\n        products may not be eligible for retu

In [50]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=128)

In [51]:
faiss_index = FAISS.from_documents(docs, embedding)

In [52]:
faiss_retriever = faiss_index.as_retriever()
results = faiss_retriever.invoke("What is the return policy?")

for result in results:
    print(result.page_content[:100])
    print(result.metadata)
    print("--------------------")


        AcmeMart offers a 30-day return policy for most products. Products must be
        unused a
{'source': 'return_policy.txt', 'category': 'returns', 'author': 'Customer Support', 'date': '2026-02-05'}
--------------------

        Refunds are initiated after the returned product passes quality inspection.
        Credit-
{'source': 'refund_policy.txt', 'category': 'payments', 'author': 'Finance Team', 'date': '2026-02-10'}
--------------------

        Premium customers receive free standard shipping, early access to selected
        sales, a
{'source': 'premium_membership.txt', 'category': 'membership', 'author': 'Marketing Team', 'date': '2026-03-05'}
--------------------

        Customers can cancel an order before it enters the shipped status.
        To cancel an ord
{'source': 'order_cancellation.txt', 'category': 'orders', 'author': 'Customer Support', 'date': '2026-02-01'}
--------------------


## PART 3 — Advanced Retrieval Strategies

**Task 4: Maximal Marginal Relevance (MMR) Retriever**
1. Use search_type="mmr".
2. Compare results with normal similarity search.
3. Explain how MMR improves diversity.

**Answer — how MMR improves diversity**
- normal similarity just picks the top-k closest chunks → often near-duplicates saying the same thing
- MMR mixes relevance + diversity: still close to the query, but not too similar to docs already picked (`lambda_mult` is the tradeoff knob)
- so you cover more of the topic instead of 3 almost-identical return-policy paragraphs


In [53]:
mmr_retriever = faiss_index.as_retriever(
    search_type="mmr",
    kwargs={
        "k": 3,
        "fetch_k": 5,   
        "lambda_mult": 0.5
    }
)

In [54]:
results = mmr_retriever.invoke("What is the return policy?")

for result in results:
    print(result.page_content[:100])
    print(result.metadata)


        AcmeMart offers a 30-day return policy for most products. Products must be
        unused a
{'source': 'return_policy.txt', 'category': 'returns', 'author': 'Customer Support', 'date': '2026-02-05'}

        Customers can cancel an order before it enters the shipped status.
        To cancel an ord
{'source': 'order_cancellation.txt', 'category': 'orders', 'author': 'Customer Support', 'date': '2026-02-01'}

        Refunds are initiated after the returned product passes quality inspection.
        Credit-
{'source': 'refund_policy.txt', 'category': 'payments', 'author': 'Finance Team', 'date': '2026-02-10'}

        Premium customers receive free standard shipping, early access to selected
        sales, a
{'source': 'premium_membership.txt', 'category': 'membership', 'author': 'Marketing Team', 'date': '2026-03-05'}


**Task 5: Multi-Query Retriever**
1. Use MultiQueryRetriever.
2. Allow LLM to generate multiple reformulated queries.
3. Retrieve documents for all queries.
4. Compare retrieval quality.

**Answer — compare retrieval quality**
- one fixed query can miss stuff if wording doesnt match the docs (“refund time” vs “return policy”)
- MultiQueryRetriever asks the LLM to rewrite the question a few ways, retrieves for each, then merges
- recall usually gets better; cost is extra LLM calls + sometimes a bit more noise


In [55]:
from langchain_classic.retrievers import MultiQueryRetriever

# Create a MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=faiss_retriever,
    llm=llm
)

In [56]:
results = multi_query_retriever.invoke("What is the return policy?")

for result in results:
    print(result.page_content[:100])
    print(result.metadata)
    print("--------------------")


        AcmeMart offers a 30-day return policy for most products. Products must be
        unused a
{'source': 'return_policy.txt', 'category': 'returns', 'author': 'Customer Support', 'date': '2026-02-05'}
--------------------

        Refunds are initiated after the returned product passes quality inspection.
        Credit-
{'source': 'refund_policy.txt', 'category': 'payments', 'author': 'Finance Team', 'date': '2026-02-10'}
--------------------

        Premium customers receive free standard shipping, early access to selected
        sales, a
{'source': 'premium_membership.txt', 'category': 'membership', 'author': 'Marketing Team', 'date': '2026-03-05'}
--------------------

        AcmeMart's privacy policy states that customer information is used to process
        order
{'source': 'privacy_policy.txt', 'category': 'privacy', 'author': 'Legal Team', 'date': '2026-03-10'}
--------------------

        Customers can cancel an order before it enters the shipped status.
        To

**Task 6: Contextual Compression Retriever**
1. Use ContextualCompressionRetriever.
2. Combine base retriever with a document compressor.
3. Reduce irrelevant content.
4. Show before vs after compression.

**Answer — before vs after**
- before: base retriever dumps whole chunks (lots of filler around the useful line)
- after: `LLMChainExtractor` keeps only the bits that actually answer the question
- shorter / cleaner context for the LLM → less noise, usually better grounded answers (pays an extra LLM pass)


In [57]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=faiss_retriever
)

In [58]:

results = compression_retriever.invoke("What is the return policy?")

for result in results:
    print(result.page_content[:100])
    print(result.metadata)
    print("--------------------")

AcmeMart offers a 30-day return policy for most products. Products must be unused and returned with 
{'source': 'return_policy.txt', 'category': 'returns', 'author': 'Customer Support', 'date': '2026-02-05'}
--------------------
Refunds are initiated after the returned product passes quality inspection. Credit-card refunds norm
{'source': 'refund_policy.txt', 'category': 'payments', 'author': 'Finance Team', 'date': '2026-02-10'}
--------------------
The membership fee is non-refundable after the 7-day cancellation period.
{'source': 'premium_membership.txt', 'category': 'membership', 'author': 'Marketing Team', 'date': '2026-03-05'}
--------------------
Customers can cancel an order before it enters the shipped status. If the order has already been shi
{'source': 'order_cancellation.txt', 'category': 'orders', 'author': 'Customer Support', 'date': '2026-02-01'}
--------------------


## PART 4 — YouTube Content RAG Chatbot (Mini Project)

**Task 7: Load YouTube Content**
1. Use YoutubeLoader to load transcript from a YouTube video.
2. Split transcript using text splitters.

In [61]:
from langchain_community.document_loaders import YoutubeLoader
import youtube_transcript_api

loader = YoutubeLoader.from_youtube_url(
    youtube_url="https://www.youtube.com/watch?v=UpE5yuhwXXc",
    language="en"
)


In [ ]:
transcript = loader.load()
print(transcript)


[Document(metadata={'source': 'UpE5yuhwXXc'}, page_content='Chuck, guess what? >> What? >> I have another explainer. >> I should have said chicken butt then. >> Not on Star Talk. >> Exactly. >> Is that joke coming out of you? No. >> Okay. We pay you too much to to be chicken button your jokes. >> I got to tell you, I would have to take uh maybe give back a few dollars [laughter] >> if I came out with guess what chicken butt. So, >> so this one I thought I\'m long overdue to do an entire explainer which might have to spill into more than one explainer >> on the periodic table of the elements. >> That\'s a whole >> It\'s a whole science. >> It\'s a science thing. [laughter] >> In fact, could you bring me my periodic table of element tie, please? Here we go. >> Oh, that\'s that\'s wild. So, I\'m not going to tell you everything about the periodic table, okay? >> Because I don\'t think all aspects of it are equally as interesting, >> okay? >> I\'m going to cherrypick >> just the stuff that

In [63]:
print(len(transcript))

1



**Task 8: Build Vector Store for YouTube Content**
1. Generate embeddings using OpenAI.
2. Store embeddings in FAISS or Chroma.
3. Create a retriever.


In [69]:
# split the transcript into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(transcript)

# create a vector store
vector_store = FAISS.from_documents(chunks, embedding)
vector_store_retriever = vector_store.as_retriever(kwargs={"k": 3})

In [70]:
results = vector_store_retriever.invoke("What is the main topic of the video?")

for result in results:
    print(result.page_content[:100])
    print(result.metadata)
    print("--------------------")


our laboratory. We have gone from element 92 to as of this recording element 118. >> 118. >> And 11 
{'source': 'UpE5yuhwXXc'}
--------------------
Chuck, guess what? >> What? >> I have another explainer. >> I should have said chicken butt then. >>
{'source': 'UpE5yuhwXXc'}
--------------------
>> There you go. That's so cool. >> That's very cool. >> And who made this up? I THAT'S WHAT I'M SAY
{'source': 'UpE5yuhwXXc'}
--------------------
just a sequence of elements in order of how many protons are in the nucleus. Period. >> Right? >> Th
{'source': 'UpE5yuhwXXc'}
--------------------



**Task 9: Build YouTube RAG Chatbot**
Create a chatbot that:
1. Accepts user questions.
2. Retrieves relevant transcript chunks.
3. Uses OpenAI LLM to generate answers.
4. Maintains conversation context.

In [68]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [71]:
prompt = ChatPromptTemplate.from_template("""
    You are an assistant answering questions about a YouTube video.
    Use ONLY the context below. If unsure, say you don't know.
    Context: {context}
    Question: {question}
    Answer:    """
)

def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

chain = {
    "context": vector_store_retriever | format_docs,
    "question": RunnablePassthrough()
} | prompt | llm | StrOutputParser()


In [72]:
answer = chain.invoke("What is the main topic of the video?")
print(answer)

The main topic of the video is the periodic table of elements, including the discovery of new elements, their properties, and the science behind how they are organized.



**Task 10: Testing & Evaluation**
1. Ask at least 5 questions related to the video.
2. Verify answers come from video content.
3. Handle unknown questions gracefully.

In [73]:
qsns = [
    "What is the main topic of the video?",
    "How does the video explain the main topic?",
    "Who is the speaker?",
    "What is the video about?",
    "Who are the guests?",
    "What is the video length?"
]

for qsn in qsns:
    answer = chain.invoke(qsn)
    print(f"Question: {qsn}")
    print(f"Answer: {answer}")
    print("--------------------")

Question: What is the main topic of the video?
Answer: The main topic of the video is the periodic table of elements, including the discovery of new elements, their properties, and the science behind how they are organized.
--------------------
Question: How does the video explain the main topic?
Answer: The video explains the main topic by discussing the periodic table of elements, the discovery of new elements, and the historical context of alchemy and quantum physics. It highlights the organization of elements by atomic number and the concept of wave duality in physics. The conversation touches on the significance of these discoveries and the properties of matter, emphasizing the importance of understanding the basics of chemistry and physics.
--------------------
Question: Who is the speaker?
Answer: I don't know.
--------------------
Question: What is the video about?
Answer: The video discusses the periodic table of elements, highlighting the discovery of new elements since 1973,

## PART 5 — Observations & Insights

**Task 11: Conceptual Questions**
Answer briefly:

1. Difference between retriever-based RAG and normal prompting → normal prompting = model answers only from what it memorized. RAG = retrieve relevant docs first, put them in the prompt, then answer — so it can use your own / fresh / long docs.
2. Why vector stores are critical → they make “find similar chunks fast” practical. without them you’d brute-force compare the query to every chunk every time.
3. When to use MMR vs similarity search → similarity when you want the single most relevant hits. MMR when top-k looks redundant and you want diverse coverage.
4. Benefits of multi-query retrieval → paraphrases the question so wording mismatch hurts less; better recall across phrasing variants.
5. Importance of contextual compression → cuts filler from retrieved chunks so the LLM sees only useful bits — less noise, shorter context, cleaner answers.
